In [67]:
import torch
import torch.nn as nn
x = torch.arange(0, 1, 0.001)
b = 10
c = -5          # boundary at x = 0.5

y = torch.sigmoid(b * x + c)
y = torch.round(y)

print(y.sum())  # should be about 500

tensor(499.)


In [68]:
from sklearn.model_selection import train_test_split

train_X, test_X, train_y, test_y = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y
)
torch.manual_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"

In [69]:
train_X[:5] , train_y[:5]

(tensor([0.8590, 0.2990, 0.7880, 0.3980, 0.5890]),
 tensor([1., 0., 1., 0., 1.]))

In [70]:
class Linear_Regression(nn.Module):
  def __init__(self):
    super().__init__()
    self.l1  = nn.Linear(in_features =1, out_features=1)

  def forward(self,x):
    return self.l1(x)

lr_model = Linear_Regression()
lr_model.to(device)


Linear_Regression(
  (l1): Linear(in_features=1, out_features=1, bias=True)
)

In [71]:
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(params= lr_model.parameters(), lr = 0.025)

def acc_fn(y_pred, y_real):
  y_pred = torch.round(torch.sigmoid(y_pred))
  return (y_pred == y_real).sum() / len(y_real)

In [72]:
epochs = 1000
lr_model.train()

for epoch in range(epochs):
  train_X = train_X.unsqueeze(dim=1)
  train_y = train_y.unsqueeze(dim=1)
  train_X = train_X.to(device)
  train_y = train_y.to(device)
  y_logit = lr_model(train_X)
  loss = loss_fn(y_logit, train_y)
  accuracy = acc_fn(y_logit, train_y)
  optimizer.zero_grad()
  loss.backward()
  optimizer.step()
  if epoch % 100 == 0:
    print("loss is" , loss.item())
    print("Accuracy is ", accuracy.item()*100)

loss is 0.7762413620948792
Accuracy is  49.87500011920929
loss is 0.4208623170852661
Accuracy is  87.37499713897705
loss is 0.29090338945388794
Accuracy is  95.37500143051147
loss is 0.2291855663061142
Accuracy is  97.12499976158142
loss is 0.19314666092395782
Accuracy is  98.1249988079071
loss is 0.1692335158586502
Accuracy is  98.50000143051147
loss is 0.1519964337348938
Accuracy is  98.87499809265137
loss is 0.13884252309799194
Accuracy is  99.00000095367432
loss is 0.12838129699230194
Accuracy is  99.00000095367432
loss is 0.11979878693819046
Accuracy is  99.12499785423279


In [74]:
#predictions on test

X_test = test_X.unsqueeze(dim=1)
y_test = test_y.unsqueeze(dim=1)
X_test = X_test.to(device)
y_test = y_test.to(device)


lr_model.eval()
with torch.inference_mode():
  y_test_pred = lr_model(X_test)
  accuracy = acc_fn(y_test_pred, y_test)
  print("Accuracy on test is ", accuracy.item()*100)


Accuracy on test is  98.50000143051147


In [78]:
lr_model.state_dict()

OrderedDict([('l1.weight', tensor([[14.5392]])),
             ('l1.bias', tensor([-7.1521]))])